In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

url = 'https://www.gutenberg.org/cache/epub/5200/pg5200.txt'
path_to_file = tf.keras.utils.get_file('kafka.txt', url)

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Length of text: {len(text)} characters')

C:\Users\mirza\anaconda3\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


Length of text: 140488 characters


In [2]:
text = text.split('*** START OF THE PROJECT GUTENBERG EBOOK')[1]
text = text.split('*** END OF THE PROJECT GUTENBERG')[0]
print(f'Length of text: {len(text)} characters')

Length of text: 120315 characters


In [3]:
text = text.replace('\r', '')

corpus = text.lower().split('\n')

In [4]:
custom_filters = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n\r“”•™’'

tokenizer = Tokenizer(filters=custom_filters)
tokenizer.fit_on_texts(corpus)

In [5]:
total_words = len(tokenizer.word_index) + 1
print(f"Total unique words: {total_words}")

Total unique words: 2643


In [6]:
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total sequences generated: {len(input_sequences)}")

Total sequences generated: 20550


In [7]:
for seq in input_sequences[:5]:
    print(seq)

[53, 151]
[53, 151, 57]
[53, 151, 57, 12]
[53, 151, 57, 12, 97]
[53, 151, 57, 12, 97, 914]


In [8]:
max_sequence_len = max([len(x) for x in input_sequences])

print(f"Max sequence length is: {max_sequence_len}")

Max sequence length is: 18


In [9]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen = max_sequence_len, padding = 'pre')

In [10]:
X = padded_input_sequences[:,:-1]
y = padded_input_sequences[:,-1]

In [11]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y, num_classes=2644)

In [12]:
y.shape

(20550, 2644)

In [13]:
for seq in y[:5]:
    print(seq)

[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]
[0. 0. 0. ... 0. 0. 0.]


In [14]:
import numpy as np

original_target = padded_input_sequences[0, -1]
categorical_target = np.argmax(y[0])

print(f"Original target word integer: {original_target}")
print(f"Index of the '1' in categorical array: {categorical_target}")

Original target word integer: 151
Index of the '1' in categorical array: 151


In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Dropout
from tensorflow.keras.metrics import Precision, Recall, AUC

In [16]:
model = Sequential()
model.add(Input(shape=(17,)))
model.add(Embedding(input_dim=2644, output_dim=100))

model.add(LSTM(150))

model.add(Dense(2644, activation='softmax'))

In [17]:
model.compile(loss='categorical_crossentropy', optimizer='adam', 
        metrics=[
            'accuracy', 
            Precision(name='precision'), 
            Recall(name='recall'), 
            AUC(name='auc')
    ])

In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 17, 100)             │         264,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 150)                 │         150,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 2644)                │         399,244 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 814,244 (3.11 MB)

 Trainable params: 814,244 (3.11 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_accuracy',  
    patience=30,               
    restore_best_weights=True,
    mode='max',
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath='best_kafka_predictor.keras',
    monitor='val_accuracy',  
    save_best_only=True,
    mode='max',             
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',     
    factor=0.5,              
    patience=15,
    min_lr=0.00001,
    verbose=1
)

my_callbacks = [early_stop, checkpoint, reduce_lr]


history = model.fit(
    X, 
    y, 
    epochs=100, 
    callbacks=my_callbacks, 
    verbose=1
)

Epoch 1/100
643/643 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0522 - auc: 0.7010 - loss: 6.1611 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 1: finished saving model to best_kafka_predictor.keras
643/643 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.0522 - auc: 0.7010 - loss: 6.1611 - precision: 0.0000e+00 - recall: 0.0000e+00 - learning_rate: 0.0010
Epoch 2/100
  1/643 ━━━━━━━━━━━━━━━━━━━━ 29s 45ms/step - accuracy: 0.0625 - auc: 0.6831 - loss: 6.2612 - precision: 0.0000e+00 - recall: 0.0000e+00

C:\Users\mirza\anaconda3\Lib\site-packages\keras\src\callbacks\early_stopping.py:100: UserWarning: Early stopping conditioned on metric `val_accuracy` which is not available. Available metrics are: accuracy,auc,loss,precision,recall
  current = self.get_monitor_value(logs)
C:\Users\mirza\anaconda3\Lib\site-packages\keras\src\callbacks\model_checkpoint.py:329: UserWarning: Can save best model only with val_accuracy available.
  if self._should_save_model(epoch, batch, logs, filepath):
C:\Users\mirza\anaconda3\Lib\site-packages\keras\src\callbacks\callback_list.py:171: UserWarning: Learning rate reduction is conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,auc,loss,precision,recall,learning_rate.
  callback.on_epoch_end(epoch, logs)


640/643 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0761 - auc: 0.7291 - loss: 5.7074 - precision: 1.0000 - recall: 1.4648e-04
Epoch 2: finished saving model to best_kafka_predictor.keras
643/643 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.0763 - auc: 0.7292 - loss: 5.7061 - precision: 1.0000 - recall: 1.4599e-04 - learning_rate: 0.0010
Epoch 3/100
640/643 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1173 - auc: 0.7600 - loss: 5.2859 - precision: 0.8641 - recall: 0.0043
Epoch 3: finished saving model to best_kafka_predictor.keras
643/643 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.1172 - auc: 0.7600 - loss: 5.2862 - precision: 0.8654 - recall: 0.0044 - learning_rate: 0.0010
Epoch 4/100
640/643 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1412 - auc: 0.7790 - loss: 4.9805 - precision: 0.8271 - recall: 0.0107
Epoch 4: finished saving model to best_kafka_predictor.keras
643/643 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.1413 - auc: 0.7790 - loss: 4.9801 - precision:

In [20]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def generate_kafka_text(seed_text, next_words, model, tokenizer, max_sequence_len):
    """
    Takes a starting phrase and generates the specified number of future words.
    """
    print(f"Generating {next_words} words...\n")
    
    for _ in range(next_words):
        
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
        
        predicted_probs = model.predict(token_list, verbose=0)
        
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]
        
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break
                
        seed_text += " " + output_word
        
    return seed_text


user_starting_text = "Gregor woke up and"

user_word_count = 30 

final_story = generate_kafka_text(
    seed_text=user_starting_text, 
    next_words=user_word_count, 
    model=model, 
    tokenizer=tokenizer, 
    max_sequence_len=18 
)

print("--- AI GENERATED TEXT ---")
print(final_story)

Generating 30 words...

--- AI GENERATED TEXT ---
Gregor woke up and had to himself the chief clerk s firm footsteps in his highly polished boots running while job would have to rush like now she would have a loud filled it
